## Diagnosis of merge and interpolation

In [82]:
# QUICK CHECK: row count + IMERG NaN spot-check on one full hour
# ============================================================================
# Fixes: expected row count updated for the buffered 3-year domain
# (21888 hours x 18471 cells), and imerg_MWobservationTime replaced with a
# dynamic check — that column was dropped from the dataset early in this
# project's correlation-based feature reduction, so referencing it directly
# would throw a KeyError. nrows also corrected to 18471 (one full hour's
# worth of grid cells in the current domain, not the old 11211).
# ============================================================================
import os
import pandas as pd

path = "merged_output_hourly_imerg/merged_data_final.csv"

print("Counting total rows (this reads the whole file, will take a while)...")
with open(path) as f:
    line_count = sum(1 for _ in f) - 1

EXPECTED_HOURS = 21888
EXPECTED_CELLS = 18471
expected = EXPECTED_HOURS * EXPECTED_CELLS

print(f"Line count: {line_count:,}")
print(f"Expected:   {expected:,}")
print(f"Match: {line_count == expected}")
if line_count != expected:
    print(f"  Difference: {abs(line_count - expected):,} rows")

print(f"\nSampling one full hour ({EXPECTED_CELLS:,} rows)...")
sample = pd.read_csv(path, nrows=EXPECTED_CELLS)

imerg_cols = [c for c in sample.columns if c.startswith("imerg_")]
print(f"IMERG columns present: {imerg_cols}\n")

print("=" * 70)
print("NaN CHECK — ONE SAMPLE HOUR")
print("=" * 70)
for col in imerg_cols:
    n_nan = sample[col].isna().sum()
    pct = 100 * n_nan / len(sample)
    print(f"  {col:<35} {n_nan:>6} NaN ({pct:.2f}%)  range: "
          f"{sample[col].min()} to {sample[col].max()}")

Counting total rows (this reads the whole file, will take a while)...
Line count: 404,293,248
Expected:   404,293,248
Match: True

Sampling one full hour (18,471 rows)...
IMERG columns present: ['imerg_precipitation', 'imerg_precipitationQualityIndex', 'imerg_probabilityLiquidPrecipitation', 'imerg_IRprecipitation', 'imerg_MWprecipSource', 'imerg_IRinfluence']

NaN CHECK — ONE SAMPLE HOUR
  imerg_precipitation                   2216 NaN (12.00%)  range: 0.0 to 15.485
  imerg_precipitationQualityIndex       2216 NaN (12.00%)  range: 0.172 to 0.63000005
  imerg_probabilityLiquidPrecipitation   2216 NaN (12.00%)  range: 15.0 to 100.0
  imerg_IRprecipitation                 5686 NaN (30.78%)  range: 0.0 to 5.955
  imerg_MWprecipSource                  2216 NaN (12.00%)  range: 0.0 to 4.5
  imerg_IRinfluence                     5215 NaN (28.23%)  range: 0.0 to 34.0


In [12]:
# CHECK: does the latitude-banded NaN pattern exist in RAW IMERG, pre-mapping?
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

GPM_DIR = "GPM_DATA"
files = [f for f in os.listdir(GPM_DIR) if f.endswith((".nc4", ".nc"))]
sample_path = os.path.join(GPM_DIR, files[0])
print(f"Checking raw file: {files[0]}")

ds_grid = xr.open_dataset(sample_path, group="Grid")
lat = ds_grid["lat"].values
lon = ds_grid["lon"].values
precip = ds_grid["precipitation"].values

UK_BOUNDS = {"lat_min": 48.5, "lat_max": 62.5, "lon_min": -9.75, "lon_max": 3.25}
lat_idx = np.where((lat >= UK_BOUNDS["lat_min"]) & (lat <= UK_BOUNDS["lat_max"]))[0]
lon_idx = np.where((lon >= UK_BOUNDS["lon_min"]) & (lon <= UK_BOUNDS["lon_max"]))[0]

precip_uk = precip[0][np.ix_(lon_idx, lat_idx)] if precip.shape[-1] == len(lat) else precip[0][np.ix_(lat_idx, lon_idx)]
lat_uk = lat[lat_idx]
lon_uk = lon[lon_idx]

nan_mask = np.isnan(precip_uk)
print(f"NaN cells in this raw file (UK domain, native IMERG grid): {nan_mask.sum():,} "
      f"/ {nan_mask.size:,} ({100*nan_mask.sum()/nan_mask.size:.2f}%)")

nan_by_lat = nan_mask.mean(axis=0) if nan_mask.shape[0] == len(lon_uk) else nan_mask.mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lat_uk, nan_by_lat * 100)
ax.set_xlabel("Latitude")
ax.set_ylabel("% NaN at this latitude")
ax.set_title("RAW IMERG (native grid, pre-mapping): NaN rate by latitude\n"
              "Spikes here = genuine IMERG characteristic. Flat here = pipeline-introduced.")
fig.tight_layout()
fig.savefig("raw_imerg_nan_by_latitude.png", dpi=150)
plt.close(fig)
print("\n✓ Saved raw_imerg_nan_by_latitude.png")
print("Compare this against the banded pattern in your merged/mapped data:")
print("if this raw plot ALSO shows spikes at the same latitudes -> genuine IMERG limitation")
print("if this raw plot is flat/uniform -> the banding is introduced by your pipeline's mapping")

Checking raw file: GPM_3IMERGHHE.07:3B-HHR-E.MS.MRG.3IMERG.20240101-S000000-E002959.0000.V07B.HDF5.dap.nc4
NaN cells in this raw file (UK domain, native IMERG grid): 0 / 18,340 (0.00%)

✓ Saved raw_imerg_nan_by_latitude.png
Compare this against the banded pattern in your merged/mapped data:
if this raw plot ALSO shows spikes at the same latitudes -> genuine IMERG limitation
if this raw plot is flat/uniform -> the banding is introduced by your pipeline's mapping


In [14]:
# CHECK: does the latitude-banded NaN pattern exist in RAW IMERG, pre-mapping?
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

GPM_DIR = "GPM_DATA"
files = [f for f in os.listdir(GPM_DIR) if f.endswith((".nc4", ".nc"))]
sample_path = os.path.join(GPM_DIR, files[0])
print(f"Checking raw file: {files[0]}")

ds_grid = xr.open_dataset(sample_path, group="Grid")
lat = ds_grid["lat"].values
lon = ds_grid["lon"].values
precip = ds_grid["precipitation"].values

UK_BOUNDS = {"lat_min": 48.5, "lat_max": 62.5, "lon_min": -9.75, "lon_max": 3.25}
lat_idx = np.where((lat >= UK_BOUNDS["lat_min"]) & (lat <= UK_BOUNDS["lat_max"]))[0]
lon_idx = np.where((lon >= UK_BOUNDS["lon_min"]) & (lon <= UK_BOUNDS["lon_max"]))[0]

precip_uk = precip[0][np.ix_(lon_idx, lat_idx)] if precip.shape[-1] == len(lat) else precip[0][np.ix_(lat_idx, lon_idx)]
lat_uk = lat[lat_idx]
lon_uk = lon[lon_idx]

nan_mask = np.isnan(precip_uk)
print(f"NaN cells in this raw file (UK domain, native IMERG grid): {nan_mask.sum():,} "
      f"/ {nan_mask.size:,} ({100*nan_mask.sum()/nan_mask.size:.2f}%)")

nan_by_lat = nan_mask.mean(axis=0) if nan_mask.shape[0] == len(lon_uk) else nan_mask.mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lat_uk, nan_by_lat * 100)
ax.set_xlabel("Latitude")
ax.set_ylabel("% NaN at this latitude")
ax.set_title("RAW IMERG (native grid, pre-mapping): NaN rate by latitude\n"
              "Spikes here = genuine IMERG characteristic. Flat here = pipeline-introduced.")
fig.tight_layout()
fig.savefig("raw_imerg_nan_by_latitude.png", dpi=150)
plt.close(fig)
print("\n✓ Saved raw_imerg_nan_by_latitude.png")
print("Compare this against the banded pattern in your merged/mapped data:")
print("if this raw plot ALSO shows spikes at the same latitudes -> genuine IMERG limitation")
print("if this raw plot is flat/uniform -> the banding is introduced by your pipeline's mapping")

Checking raw file: GPM_3IMERGHHE.07:3B-HHR-E.MS.MRG.3IMERG.20240101-S000000-E002959.0000.V07B.HDF5.dap.nc4
NaN cells in this raw file (UK domain, native IMERG grid): 0 / 18,340 (0.00%)

✓ Saved raw_imerg_nan_by_latitude.png
Compare this against the banded pattern in your merged/mapped data:
if this raw plot ALSO shows spikes at the same latitudes -> genuine IMERG limitation
if this raw plot is flat/uniform -> the banding is introduced by your pipeline's mapping


In [83]:
# Dedup by hour block, without a full-file sort — much faster
import os

seen_hours = set()
kept = 0
total = 0
header_written = False
source_file = "merged_output_hourly_imerg/merged_data_final.csv"
final_file = "merged_output_hourly_imerg/merged_data_final_dedupped.csv"
with open(source_file) as fin, open(final_file, 'w') as fout:
    header = fin.readline()
    fout.write(header)

    current_hour = None
    current_block = []
    skip_current = False

    for line in fin:
        total += 1
        t = line.split(',', 1)[0]

        if t != current_hour:
            # flush previous block if it wasn't a skip
            if current_hour is not None and not skip_current:
                fout.writelines(current_block)
                kept += len(current_block)
            # start new block
            current_hour = t
            current_block = []
            skip_current = (t in seen_hours) or t.startswith('2026')
            seen_hours.add(t)

        if not skip_current:
            current_block.append(line)

    # flush the last block
    if current_hour is not None and not skip_current:
        fout.writelines(current_block)
        kept += len(current_block)

print(f"✓ Dedup complete (no sort needed): {total:,} scanned, {kept:,} kept")

expected = 17544 * 11211
print(f"Expected: {expected:,}, Actual: {kept:,}, Match: {kept == expected}")

KeyboardInterrupt: 

## NaN counts

In [ ]:

print("NaN count in era5_t2m:", sample['era5_t2m'].isna().sum())
print("NaN count in era5_sst:", sample['era5_sst'].isna().sum(), "(expect non-zero — legitimate land mask)")

NaN count in era5_t2m: 0
NaN count in era5_sst: 4729 (expect non-zero — legitimate land mask)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('eda_output', exist_ok=True)

csv_file = "merged_output_hourly_imerg/merged_data_final.csv"

# ── 1. FULL-DATASET NaN COUNT PER COLUMN ──
print("Scanning full dataset for NaN counts per column...")
chunk_iter = pd.read_csv(csv_file, chunksize=1_000_000)
nan_counts = {}
total_rows = 0

for chunk in chunk_iter:
    total_rows += len(chunk)
    for col in chunk.columns:
        if col == 'time':
            continue
        n = chunk[col].isna().sum()
        nan_counts[col] = nan_counts.get(col, 0) + n

print(f"\nTotal rows scanned: {total_rows:,}\n")
print(f"{'Column':<35} {'NaN Count':>15} {'NaN %':>10}")
print("-" * 62)
for col, n in sorted(nan_counts.items(), key=lambda x: -x[1]):
    pct = 100 * n / total_rows
    flag = "  ⚠" if n > 0 else ""
    print(f"{col:<35} {n:>15,} {pct:>9.3f}%{flag}")

# ── 2. OUT-OF-BOUNDS CHECK (lat/lon within domain) ──
print("\n" + "="*70)
print("OUT-OF-BOUNDS CHECK (lat/lon)")
print("="*70)
UK_BOUNDS = {"lat_min": 48.4, "lat_max": 62.4, "lon_min": -9.7, "lon_max": 3.4}

lat_lon_check = pd.read_csv(csv_file, usecols=['latitude', 'longitude'], chunksize=1_000_000)
oob_count = 0
for chunk in lat_lon_check:
    oob = chunk[
        (chunk['latitude'] < UK_BOUNDS['lat_min']) | (chunk['latitude'] > UK_BOUNDS['lat_max']) |
        (chunk['longitude'] < UK_BOUNDS['lon_min']) | (chunk['longitude'] > UK_BOUNDS['lon_max'])
    ]
    oob_count += len(oob)

print(f"Out-of-bounds rows: {oob_count:,}")
print("✓ PASS — no out-of-bounds coordinates" if oob_count == 0 else "✗ FAIL — investigate")

# ── 3. IF ANY COLUMN HAS NaN, PLOT ITS SPATIAL PATTERN ──
cols_with_nan = [c for c, n in nan_counts.items() if n > 0]
print(f"\nColumns with any NaN: {cols_with_nan if cols_with_nan else 'NONE'}")

if cols_with_nan:
    for col in cols_with_nan:
        print(f"\nMapping NaN pattern for '{col}'...")
        chunk_iter2 = pd.read_csv(csv_file, usecols=['time', 'latitude', 'longitude', col],
                                   chunksize=1_000_000)
        nan_cells = set()
        for chunk in chunk_iter2:
            nan_rows = chunk[chunk[col].isna()]
            cells = set(zip(nan_rows['latitude'].round(4), nan_rows['longitude'].round(4)))
            nan_cells.update(cells)

        if nan_cells:
            lats = [c[0] for c in nan_cells]
            lons = [c[1] for c in nan_cells]
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.scatter(lons, lats, s=15, c='red', label=f'{len(nan_cells)} NaN cells')
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')
            ax.set_title(f"NaN cells for '{col}' (unique locations across all hours)")
            ax.legend()
            ax.grid(True, alpha=0.3)
            safe_name = col.replace('/', '_')
            plt.savefig(f'eda_output/nan_cells_{safe_name}.png', dpi=120, bbox_inches='tight')
            plt.show()
            print(f"  Saved eda_output/nan_cells_{safe_name}.png ({len(nan_cells)} unique locations)")
else:
    print("\n✓ No NaN values found in ANY column — dataset is fully clean.")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Total rows: {total_rows:,}")
print(f"Out-of-bounds coordinates: {oob_count:,}")
print(f"Columns with NaN: {len(cols_with_nan)}")

In [17]:
# DROP IR COLUMNS FROM merged_data_final.csv
# ============================================================================
# Removes imerg_IRprecipitation and imerg_IRinfluence, which have 28-34%
# structurally missing values due to satellite IR retrieval geometry — too
# high to handle cleanly for a spatiotemporal grid model, and redundant with
# imerg_precipitation (which already incorporates IR contributions) and
# imerg_precipitationQualityIndex (which already captures estimate reliability).
# Uses the same safe temp-file + atomic-replace pattern used throughout this
# project — never overwrites the source until the full write is verified.
# ============================================================================
import os
import pandas as pd

TARGET_PATH = "merged_output_hourly_imerg/merged_data_final.csv"
TEMP_PATH = "merged_output_hourly_imerg/merged_data_final.tmp.csv"
CHUNK_SIZE = 1_000_000

COLS_TO_DROP = ["imerg_IRprecipitation", "imerg_IRinfluence"]

header_cols = pd.read_csv(TARGET_PATH, nrows=0).columns.tolist()
actually_present = [c for c in COLS_TO_DROP if c in header_cols]
already_gone = [c for c in COLS_TO_DROP if c not in header_cols]

if already_gone:
    print(f"Note: already absent (no action needed): {already_gone}")
if not actually_present:
    print("✓ All target columns already removed — nothing to do.")
else:
    print(f"Dropping: {actually_present}")
    keep_cols = [c for c in header_cols if c not in COLS_TO_DROP]
    print(f"Keeping {len(keep_cols)} of {len(header_cols)} columns\n")
    print(f"Source: {TARGET_PATH} ({os.path.getsize(TARGET_PATH)/1e9:.2f} GB)")

    n_rows = 0
    first_chunk = True
    reader = pd.read_csv(TARGET_PATH, chunksize=CHUNK_SIZE, on_bad_lines="skip")
    for i, chunk in enumerate(reader):
        chunk = chunk.drop(columns=actually_present, errors="ignore")
        chunk.to_csv(TEMP_PATH, mode="w" if first_chunk else "a",
                     header=first_chunk, index=False)
        first_chunk = False
        n_rows += len(chunk)
        if (i + 1) % 20 == 0:
            print(f"  ... {n_rows:,} rows written")

    print(f"\n✓ Wrote {n_rows:,} rows to {TEMP_PATH}")

    out_header = pd.read_csv(TEMP_PATH, nrows=0).columns.tolist()
    remaining_ir = [c for c in COLS_TO_DROP if c in out_header]
    if remaining_ir:
        print(f"⚠ IR columns still present in output: {remaining_ir} — NOT replacing source.")
        os.remove(TEMP_PATH)
    else:
        print(f"✓ Verified: IR columns absent from output")
        print(f"  Output columns ({len(out_header)}): {out_header}")
        os.replace(TEMP_PATH, TARGET_PATH)
        print(f"\n✓ Replaced {TARGET_PATH} in place — IR columns removed")
        print(f"  New file size: {os.path.getsize(TARGET_PATH)/1e9:.2f} GB")

Dropping: ['imerg_IRprecipitation', 'imerg_IRinfluence']
Keeping 26 of 28 columns

Source: merged_output_hourly_imerg/merged_data_final.csv (96.25 GB)
  ... 20,000,000 rows written
  ... 40,000,000 rows written
  ... 60,000,000 rows written
  ... 80,000,000 rows written
  ... 100,000,000 rows written
  ... 120,000,000 rows written
  ... 140,000,000 rows written
  ... 160,000,000 rows written
  ... 180,000,000 rows written
  ... 200,000,000 rows written
  ... 220,000,000 rows written
  ... 240,000,000 rows written
  ... 260,000,000 rows written
  ... 280,000,000 rows written
  ... 300,000,000 rows written
  ... 320,000,000 rows written
  ... 340,000,000 rows written
  ... 360,000,000 rows written
  ... 380,000,000 rows written
  ... 400,000,000 rows written

✓ Wrote 404,293,248 rows to merged_output_hourly_imerg/merged_data_final.tmp.csv
✓ Verified: IR columns absent from output
  Output columns (26): ['time', 'latitude', 'longitude', 'era5_u10', 'era5_v10', 'era5_d2m', 'era5_t2m', 'era

## Creating wind vector, removing u-comp v-comp attributes

In [18]:
# ADD wind_speed10, DROP era5_u10/era5_v10 — IN-PLACE, MEMORY-SAFE, CHUNKED
# ============================================================================
# Derives wind_speed10 = sqrt(era5_u10^2 + era5_v10^2) and drops the raw
# components in one chunked pass. Writes to a temp file first, verifies the
# output is correct, then atomically replaces the source — same safe pattern
# used throughout this project for in-place column rewrites.
# ============================================================================
import os
import time
import numpy as np
import pandas as pd

CLEANED_DIR = "merged_output_hourly_imerg"
TARGET_PATH = os.path.join(CLEANED_DIR, "merged_data_final.csv")
TEMP_PATH = os.path.join(CLEANED_DIR, "merged_data_final.tmp.csv")

CHUNK_SIZE = 1_000_000
U_COL = "era5_u10"
V_COL = "era5_v10"
WIND_COL = "wind_speed10"

header_cols = pd.read_csv(TARGET_PATH, nrows=0).columns.tolist()
if U_COL not in header_cols or V_COL not in header_cols:
    candidates = [c for c in header_cols if any(k in c.lower() for k in ["wind", "u10", "v10"])]
    raise KeyError(f"Expected '{U_COL}' and '{V_COL}' not found.\n"
                   f"Wind-related candidates: {candidates}\n"
                   f"Full columns: {header_cols}")

if WIND_COL in header_cols:
    print(f"✓ '{WIND_COL}' already present — nothing to do.")
else:
    print(f"Source: {TARGET_PATH} ({os.path.getsize(TARGET_PATH)/1e9:.2f} GB)")
    print(f"Adding '{WIND_COL}' = sqrt({U_COL}^2 + {V_COL}^2), dropping raw components\n")

    t0 = time.time()
    n_rows = 0
    first_chunk = True

    reader = pd.read_csv(TARGET_PATH, chunksize=CHUNK_SIZE, on_bad_lines="skip")
    for i, chunk in enumerate(reader):
        u = pd.to_numeric(chunk[U_COL], errors="coerce")
        v = pd.to_numeric(chunk[V_COL], errors="coerce")
        chunk[WIND_COL] = np.sqrt(u**2 + v**2).astype("float32")
        chunk = chunk.drop(columns=[U_COL, V_COL])

        chunk.to_csv(TEMP_PATH, mode="w" if first_chunk else "a",
                     header=first_chunk, index=False)
        first_chunk = False
        n_rows += len(chunk)

        if (i + 1) % 20 == 0:
            elapsed = time.time() - t0
            print(f"  ... {n_rows:,} rows, {elapsed:,.1f}s elapsed")

    print(f"\n✓ Wrote {n_rows:,} rows in {time.time()-t0:,.1f}s")

    out_cols = pd.read_csv(TEMP_PATH, nrows=0).columns.tolist()
    if WIND_COL not in out_cols:
        print(f"⚠ '{WIND_COL}' missing from output — NOT replacing source.")
        os.remove(TEMP_PATH)
    elif U_COL in out_cols or V_COL in out_cols:
        print(f"⚠ Raw components still present in output — NOT replacing source.")
        os.remove(TEMP_PATH)
    else:
        os.replace(TEMP_PATH, TARGET_PATH)
        print(f"✓ Replaced {TARGET_PATH} in place")
        print(f"  Added: '{WIND_COL}'  |  Removed: '{U_COL}', '{V_COL}'")
        print(f"  Columns ({len(out_cols)}): {out_cols}")
        print(f"  New file size: {os.path.getsize(TARGET_PATH)/1e9:.2f} GB")

Source: merged_output_hourly_imerg/merged_data_final.csv (93.49 GB)
Adding 'wind_speed10' = sqrt(era5_u10^2 + era5_v10^2), dropping raw components

  ... 20,000,000 rows, 187.6s elapsed
  ... 40,000,000 rows, 376.2s elapsed
  ... 60,000,000 rows, 563.4s elapsed
  ... 80,000,000 rows, 751.0s elapsed
  ... 100,000,000 rows, 938.0s elapsed
  ... 120,000,000 rows, 1,125.0s elapsed
  ... 140,000,000 rows, 1,313.4s elapsed
  ... 160,000,000 rows, 1,502.1s elapsed
  ... 180,000,000 rows, 1,689.0s elapsed
  ... 200,000,000 rows, 1,876.3s elapsed
  ... 220,000,000 rows, 2,066.6s elapsed
  ... 240,000,000 rows, 2,257.3s elapsed
  ... 260,000,000 rows, 2,449.0s elapsed
  ... 280,000,000 rows, 2,640.4s elapsed
  ... 300,000,000 rows, 2,832.4s elapsed
  ... 320,000,000 rows, 3,033.4s elapsed
  ... 340,000,000 rows, 3,241.4s elapsed
  ... 360,000,000 rows, 3,442.8s elapsed
  ... 380,000,000 rows, 3,634.1s elapsed
  ... 400,000,000 rows, 3,826.2s elapsed

✓ Wrote 404,293,248 rows in 3,867.5s
✓ Replac

## Scanning for continuity

In [19]:
# TIME CONTINUITY CHECK PER GRID CELL
# ============================================================================
# Confirms whether each 0.1° grid cell has a gap-free hourly time series.
# This matters before building 6h-ahead labels: a naive shift(6) per cell is
# only valid if there really are no missing hours in between — otherwise
# "6 rows later" silently stops meaning "6 hours later" for some cells.
#
# Updates from original:
#   1. CSV_PATH: now reads from merged_data_final.csv (current in-place file)
#   2. UK_BOUNDS: updated to the buffered domain (48.5-62.5N, -9.75-3.25E)
#   3. n_lat/n_lon: added the +1 fencepost fix established earlier in this
#      project (round((max-min)/step) undercounts by 1 — a real bug that
#      previously caused incorrect grid indexing in several scripts)
# ============================================================================
import os
import time
import numpy as np
import pandas as pd

CLEANED_DIR = "merged_output_hourly_imerg"
CSV_PATH = os.path.join(CLEANED_DIR, "merged_data_final.csv")

UK_BOUNDS = {"lat_min": 48.5, "lat_max": 62.5, "lon_min": -9.75, "lon_max": 3.25}
GRID_RES = 0.1
CHUNK_SIZE = 1_000_000

# +1 required — round((max-min)/step) counts intervals, not points
n_lat = int(round((UK_BOUNDS["lat_max"] - UK_BOUNDS["lat_min"]) / GRID_RES)) + 1
n_lon = int(round((UK_BOUNDS["lon_max"] - UK_BOUNDS["lon_min"]) / GRID_RES)) + 1
print(f"Grid: {n_lat} lat x {n_lon} lon = {n_lat * n_lon:,} cells expected")

cell_hours = {}

print(f"\nScanning {CSV_PATH} for time continuity per grid cell...")
t0 = time.time()
n_rows = 0

reader = pd.read_csv(
    CSV_PATH,
    usecols=["time", "latitude", "longitude"],
    chunksize=CHUNK_SIZE,
    on_bad_lines="skip",
)

for i, chunk in enumerate(reader):
    chunk = chunk.dropna(subset=["time", "latitude", "longitude"])
    if len(chunk) == 0:
        continue
    n_rows += len(chunk)

    lat = chunk["latitude"].to_numpy(dtype=np.float64)
    lon = chunk["longitude"].to_numpy(dtype=np.float64)
    lat_idx = np.clip(
        np.round((lat - UK_BOUNDS["lat_min"]) / GRID_RES).astype(np.int64),
        0, n_lat - 1
    )
    lon_idx = np.clip(
        np.round((lon - UK_BOUNDS["lon_min"]) / GRID_RES).astype(np.int64),
        0, n_lon - 1
    )
    cell_key = lat_idx * n_lon + lon_idx

    time_str = chunk["time"].astype(str)
    date_only = time_str.str.len() == 10
    time_str = time_str.where(~date_only, time_str + " 00:00:00")
    parsed = pd.to_datetime(time_str, format="%Y-%m-%dT%H:%M:%S", errors="coerce")
    if parsed.isna().any():
        parsed = pd.to_datetime(time_str, format="mixed", errors="coerce")
    t_hours = (
        (parsed - pd.Timestamp("1970-01-01")) // pd.Timedelta(hours=1)
    ).to_numpy(dtype=np.int64)

    df = pd.DataFrame({"cell": cell_key, "hour": t_hours})
    for key, group in df.groupby("cell", sort=False)["hour"]:
        cell_hours.setdefault(key, []).append(group.to_numpy())

    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        print(f"  Scanned {i+1} chunks, {n_rows:,} rows, {elapsed:,.1f}s elapsed")

print(f"✓ Scanned {n_rows:,} rows in {time.time() - t0:,.1f}s. Checking continuity...")

n_cells = 0
n_fully_continuous = 0
gap_size_counts = {}
cells_with_gaps = 0

for key, arrays in cell_hours.items():
    hours = np.unique(np.concatenate(arrays))
    n_cells += 1
    if len(hours) < 2:
        continue
    diffs = np.diff(hours)
    if np.all(diffs == 1):
        n_fully_continuous += 1
    else:
        cells_with_gaps += 1
        for gap in diffs[diffs != 1]:
            gap_size_counts[int(gap)] = gap_size_counts.get(int(gap), 0) + 1

print(f"\nGrid cells with any data: {n_cells:,}")
print(f"Fully continuous (1h steps, no gaps): {n_fully_continuous:,} "
      f"({100*n_fully_continuous/max(n_cells,1):.1f}%)")
print(f"Cells with at least one gap: {cells_with_gaps:,} "
      f"({100*cells_with_gaps/max(n_cells,1):.1f}%)")
print("\nGap size distribution (hours skipped, top 15):")
for gap, cnt in sorted(gap_size_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {gap}h gap: {cnt:,} occurrences")

Grid: 141 lat x 131 lon = 18,471 cells expected

Scanning merged_output_hourly_imerg/merged_data_final.csv for time continuity per grid cell...
  Scanned 20 chunks, 20,000,000 rows, 17.5s elapsed
  Scanned 40 chunks, 40,000,000 rows, 34.6s elapsed
  Scanned 60 chunks, 60,000,000 rows, 51.4s elapsed
  Scanned 80 chunks, 80,000,000 rows, 68.3s elapsed
  Scanned 100 chunks, 100,000,000 rows, 85.3s elapsed
  Scanned 120 chunks, 120,000,000 rows, 102.5s elapsed
  Scanned 140 chunks, 140,000,000 rows, 119.3s elapsed
  Scanned 160 chunks, 160,000,000 rows, 136.5s elapsed
  Scanned 180 chunks, 180,000,000 rows, 153.5s elapsed
  Scanned 200 chunks, 200,000,000 rows, 170.5s elapsed
  Scanned 220 chunks, 220,000,000 rows, 187.5s elapsed
  Scanned 240 chunks, 240,000,000 rows, 204.7s elapsed
  Scanned 260 chunks, 260,000,000 rows, 221.9s elapsed
  Scanned 280 chunks, 280,000,000 rows, 239.1s elapsed
  Scanned 300 chunks, 300,000,000 rows, 256.0s elapsed
  Scanned 320 chunks, 320,000,000 rows, 273.

In [ ]:
import pandas as pd
import numpy as np

# Find the most extreme era5_cp values and check when/where they occurred
df = pd.read_csv("merged_output_hourly_imerg/merged_data_normalised.csv",
                 usecols=["time", "latitude", "longitude", "era5_cp"],
                 nrows=1_000_000)
top = df.nlargest(10, "era5_cp")
print(top)